# `uncertain` — a playground for the measure axesEvery measure in this library is one configuration of a single expression. You pick a**unit**, a **horizon**, a **representation**, a **distance**, a **set statistic**, a**warping** and a **sampler**, and the engine estimates the result — surprisal, entropy andinformation value are just particular settings of those seven knobs.This notebook exists so you can turn the knobs and watch the number move.**How to use it:** run the cells top to bottom once. Then go back to the cell marked**THE KNOBS**, change values, and re-run just that cell. Everything after it is aone-axis-at-a-time comparison you can also edit freely.Importing the package performs no downloads, and everything up to the final section runs ona tiny built-in toy model in milliseconds. The GPT-2 section at the end is optional.

## 1. Setup

In [ ]:
import numpy as npimport pandas as pdfrom uncertain import Measure, ToyBackend, Stimulus, Sweep, ConfigErrorfrom uncertain.registry import _REGISTRYprint("uncertain", __import__("uncertain").__version__)# A tiny exact finite model: no download, no GPU, deterministic.# `vocabulary` pieces concatenate when decoded, so " y" starts a new word.backend = ToyBackend(vocabulary=("x", " y", " z"),                     transitions={(): (.5, .3, .2)})# A Stimulus is a (context, target) pair of token ids. The target is what was# actually observed; the measure compares it against what the model expected.stimulus = Stimulus(context=backend.encode("x"), target=backend.encode(" y"))print("context:", stimulus.context, "target:", stimulus.target)

### What is available on each axisRead this off the registries rather than trusting a doc comment — it is always current.

In [ ]:
for kind in ("unit", "encoder", "distance", "statistic", "warping", "sampler"):    names = sorted(n for n, _ in _REGISTRY[kind])    print(f"{kind:10s} {names if names else '(none registered; samplers are injected objects)'}")

## 2. THE KNOBS**This is the cell to play with.** Change any value, re-run, watch the number move.Invalid combinations raise `ConfigError` before any model work happens — that is by design,so the cell catches it and prints the reason instead of a traceback.

In [ ]:
UNIT           = "token"        # "token" | "word" | "sentence"HORIZON        = 1              # H >= 1, counted in UNIT (not in tokens)REPRESENTATION = "tokens"       # "tokens" | "strings" | "layer:8" (layer needs a real model)DISTANCE       = "discrete"     # discrete | prefix_indicator | ngram_overlap | ngram_distance                                # cosine | euclidean  (these two need a vector representation)STATISTIC      = "energy"       # "energy" | "mean" | "min"WARPING        = "identity"     # "identity" | "neglog"SHORT_SAMPLES  = "compare_ragged"   # "compare_ragged" | "pad" | "drop"ESTIMATOR      = "auto"         # "auto" | "mc" | "exact"N              = 256            # number of sampled alternativesSEED           = 0MODE           = "observed"     # "observed" | "entropy" | "anticipatory"def describe(result, label=""):    """Print everything a Result carries. It is never a bare float."""    se = "n/a" if result.se is None else f"{result.se:.5f}"    ci = ("n/a" if result.ci_low is None          else f"[{result.ci_low:.5f}, {result.ci_high:.5f}]")    print(f"{label}value     = {result.value:.6f}")    print(f"{' ' * len(label)}se        = {se}      ci = {ci}")    print(f"{' ' * len(label)}method    = {result.method}   status = {result.status}")    print(f"{' ' * len(label)}n         = {result.n_requested} requested, "          f"{result.n_a_used} used on A, {result.n_b_used} on B")    d = result.diagnostics    print(f"{' ' * len(label)}truncation= {d.get('truncation_bound')} ({d.get('truncation_status')})")    print(f"{' ' * len(label)}spec_hash = {result.spec_hash[:16]}...")def build(**overrides):    """Build a Measure from the knobs above, with optional per-call overrides."""    cfg = dict(unit=UNIT, horizon=HORIZON, representation=REPRESENTATION,               distance=DISTANCE, statistic=STATISTIC, warping=WARPING,               short_samples=SHORT_SAMPLES, estimator=ESTIMATOR,               n=N, seed=SEED, mode=MODE)    cfg.update(overrides)    return Measure.from_config(backend, **cfg)try:    measure = build()    describe(measure.compute(stimulus))except ConfigError as exc:    print("ConfigError (raised before any sampling):\n   ", exc)

### Why some combinations are refusedThe library type-checks the configuration *before* it samples anything, so a bad pairingcosts milliseconds rather than a GPU job. Two rules cause most refusals:- **`energy` needs a negative-type dissimilarity.** Mean pairwise distance stays above zero  even when the two distributions are identical, because it still contains the spread of the  distribution itself. Energy distance subtracts that off — but only a distance with the  right mathematical property makes that valid.- **The representation and the distance must agree on a type.** `tokens` produces unit  strings; `cosine` wants vectors. Pairing them is refused rather than coerced.The grid below is computed live, so it cannot drift out of date.

In [ ]:
rows = []for dist in ("prefix_indicator", "ngram_overlap", "ngram_distance", "discrete"):    row = {"distance": dist}    for stat in ("mean", "min", "energy"):        try:            Measure.from_config(backend, representation="tokens", distance=dist,                                statistic=stat, n=8)            row[stat] = "ok"        except ConfigError as exc:            row[stat] = "refused" if "negative-type" in str(exc) else "refused*"    rows.append(row)print(pd.DataFrame(rows).to_string(index=False))print("\nRepresentation -> distance type compatibility:")for rep, dists in (("tokens  ", "prefix_indicator, ngram_overlap, ngram_distance, discrete"),                   ("layer:N ", "cosine, euclidean  (needs an HF backend, see section 8)"),                   ("strings ", "no core distance consumes strings - plugin territory")):    print(f"  {rep} -> {dists}")

## 3. Horizon — how far ahead the comparison looks`H` counts **units**, not tokens. Generation is token-by-token, but for `unit="word"` thesampler keeps going until it has seen `H` word boundaries.Watch the cost as well as the value: `H=1` on token units has a closed form, so the librarysolves it exactly and reports `se = 0`. Longer horizons fall back to Monte Carlo.

In [ ]:
for h in (1, 2, 3, 5):    r = build(horizon=h).compute(stimulus)    se = "exact" if r.se == 0 else f"se={r.se:.4f}"    print(f"H={h}  value={r.value: .5f}  {se:>12s}  method={r.method}")

## 4. Distance and statistic — what "different" meansThe distance `g` scores two individual alternatives. The statistic `s` reduces the wholepair of sets to one number. Keep everything else fixed and vary them.

In [ ]:
out = []for dist in ("discrete", "prefix_indicator", "ngram_overlap", "ngram_distance"):    for stat in ("mean", "min", "energy"):        try:            r = build(horizon=3, distance=dist, statistic=stat, estimator="mc").compute(stimulus)            out.append({"distance": dist, "statistic": stat, "value": round(r.value, 5),                        "se": None if r.se is None else round(r.se, 5), "status": r.status})        except ConfigError:            out.append({"distance": dist, "statistic": stat, "value": None,                        "se": None, "status": "refused at config time"})print(pd.DataFrame(out).to_string(index=False))print()print("Note: `min` reads 0 here because with a small vocabulary some sampled")print("alternative always matches the target exactly. It is a directed statistic -")print("B queries A - and it reports no uncertainty, hence status/se of `n/a`.")

### Why energy is the defaultDraw two sets from the *same* predictive distribution. A good divergence should read zero.Mean pairwise distance does not — it still contains the entropy of the distribution, so itreports spread under the name of divergence. Energy distance corrects for exactly that.

In [ ]:
from uncertain.registry import resolvefrom uncertain import SamplingSpec, UnitSpec, EstimatorSpecfrom uncertain.samplers import Samplerfrom uncertain.statistics import Mean, Energycoin = ToyBackend(transitions={(): (.5, .5)})       # vocabulary ("a", "b")sampler, encoder = Sampler(coin), resolve("encoder", "tokens")A = encoder.encode(sampler.sample((), 1, SamplingSpec(coin.model_spec, 400, 1), UnitSpec()), "compare_ragged")B = encoder.encode(sampler.sample((), 1, SamplingSpec(coin.model_spec, 400, 2), UnitSpec()), "compare_ragged")discrete, est = resolve("distance", "discrete"), EstimatorSpec(method="mc", bootstrap=99)energy = Energy().estimate(A, B, discrete, est, (11, 12))mean   = Mean().estimate(A, B, discrete, est, (11, 12))print("two independent draws from the SAME distribution (p=0.5):")print(f"  energy distance      = {energy.value: .5f}  (se {energy.se:.5f})  -> indistinguishable from 0")print(f"  mean pairwise dist.  = {mean.value: .5f}  (se {mean.se:.5f})  -> pinned near 2p(1-p) = 0.5")

## 5. Warping — the `f` on the outside`identity` reports the statistic itself. `neglog` turns a probability-like quantity into asurprisal in nats.A non-affine warping does not commute with averaging: `f` of the mean is not the mean of`f`. The library reports that gap as `warping_bias` instead of quietly ignoring it, so tworuns that differ only in `n` stay comparable.

In [ ]:
for warp in ("identity", "neglog"):    r = build(horizon=2, distance="prefix_indicator", statistic="mean",              warping=warp, estimator="mc").compute(stimulus)    print(f"{warp:9s} value={r.value: .5f}  raw={r.raw_value: .5f}  "          f"warping_bias={r.warping_bias}")

## 6. Unit — token, word, sentenceThe unit is fixed for a whole run, and horizon, granularity and alignment are all expressedin it. `word` and `sentence` need a vocabulary whose pieces contain whitespace, which is whythis toy backend spells its vocabulary `("x", " y", " z")`.

In [ ]:
for unit in ("token", "word", "sentence"):    try:        r = build(unit=unit, horizon=2, statistic="mean", estimator="mc").compute(stimulus)        print(f"{unit:9s} value={r.value: .5f}  se={r.se:.5f}")    except ConfigError as exc:        print(f"{unit:9s} refused: {exc}")# Short samples - those that hit EOS or the token budget before completing H units -# must be handled by a policy you declare. It changes the number, so it is never a default.# This backend has an EOS piece, so a good fraction of draws stop early.ends = ToyBackend(vocabulary=("x", " y", ""), transitions={(): (.45, .35, .20)},                  eos_token_id=2)print()for policy in ("compare_ragged", "pad", "drop"):    m = Measure.from_config(ends, unit="word", horizon=3, statistic="mean",                            distance="ngram_distance", short_samples=policy,                            estimator="mc", n=256, seed=0)    r = m.compute(Stimulus(context=(0,), target=(1,)))    print(f"short_samples={policy:15s} value={r.value: .5f}  n_a_used={r.n_a_used:3d}  "          f"eos_fraction={r.diagnostics['eos_fraction']:.2f}  "          f"truncation_bound={r.diagnostics['truncation_bound']:.4f}")print()print("  compare_ragged ignores the missing units; pad marks them with a symbol the")print("  target never contains, pushing short draws further away; drop removes those")print("  draws entirely - note n_a_used falls. Because samples really are cut short here,")print("  `truncation_bound` is non-zero: mass beyond the horizon that the estimate misses.")

## 7. Sampler — and the sweep that reuses its workThe sampler draws alternatives. Its defaults are the raw model distribution: temperature 1,`top_k=0`, `top_p=1`, no truncation and no first-step EOS suppression — nothing is inheritedfrom a generation library's opinions.The important property is that a sweep **regenerates nothing**. The predictive state afterobserving position `t` is the state before position `t+1`, so one sample set per positionserves the whole sentence, and every cell of the grid is a reduction over those samesamples. That also makes the comparisons paired.

In [ ]:
grid = Sweep(base=build(horizon=2, statistic="mean", estimator="mc"),             horizon=[1, 2, 3],             distance=["discrete", "ngram_distance"],             statistic=["mean", "min"])items = [Stimulus(context=backend.encode("x"), target=backend.encode(" y"), stimulus_id="i1"),         Stimulus(context=backend.encode(" y"), target=backend.encode(" z"), stimulus_id="i2")]plan = grid.plan(items)print("plan:", plan.describe())print("  -> cells x items rows, but only `sample_requests` generation passes\n")df = grid.run(items).to_dataframe()print(df[["stimulus_id", "horizon", "distance", "statistic", "value", "se", "n_a_used"]]      .to_string(index=False))

In [ ]:
# Sampling knobs live on the sampler, and changing them changes the measure's identity.base = build(statistic="mean", estimator="mc", horizon=2)print("default generation:", dict(base.spec.sampling.generation))print("spec_hash:", base.spec_hash[:16], "...")warmer = Measure.from_config(backend, unit=UNIT, horizon=2, statistic="mean",                             estimator="mc", n=N, seed=SEED,                             generation={"temperature": 1.5}, probability="sampling")print("\nwarmer generation:", dict(warmer.spec.sampling.generation))print("spec_hash:", warmer.spec_hash[:16], "...  <- a different measure, not a tweak")print("\nvalue at temperature 1.0:", round(base.compute(stimulus).value, 5))print("value at temperature 1.5:", round(warmer.compute(stimulus).value, 5))

## 8. Optional: a real modelEverything above runs on the toy backend. The same seven knobs drive a real language model —this is the only cell that needs GPT-2 on disk, and it is the only way to use the `layer:N`representation, since that axis needs hidden states.Skip this section if you just want to play with the axes.

In [ ]:
RUN_GPT2 = True     # set False to skipif RUN_GPT2:    from uncertain import HFBackend    gpt2 = HFBackend("openai-community/gpt2",                     revision="607a30d783dfa663caf39e06633721c8d4cfcd7e")    ctx = tuple(gpt2.encode("The capital of France is"))    tgt = tuple(gpt2.encode(" Paris"))    item = Stimulus(context=ctx, target=tgt)    surprisal = Measure.from_config(gpt2, unit="token", horizon=1, representation="tokens",                                    distance="prefix_indicator", statistic="mean",                                    warping="neglog", n=64)    r = surprisal.compute(item)    print(f"surprisal of ' Paris' given 'The capital of France is' = {r.value:.4f} nats "          f"({r.method})")    # The representation axis: compare an early layer against a middle one.    for layer in ("layer:0", "layer:8"):        m = Measure.from_config(gpt2, unit="token", horizon=3, representation=layer,                                distance="cosine", statistic="energy", n=16)        r = m.compute(item)        print(f"{layer}  cosine/energy  value={r.value:.5f}  se={r.se:.5f}")else:    print("skipped")

## 9. What you get backA `Result` is never a bare float. It carries the estimate, its standard error and confidenceinterval, the `n` that produced it, a status, the diagnostics, and a `spec_hash` thatidentifies the exact configuration that produced it — which is what makes a numberreproducible and citable later.

In [ ]:
r = build(horizon=3, statistic="mean", estimator="mc").compute(stimulus)describe(r, label="")print("\nfull diagnostics:")for k, v in r.diagnostics.items():    text = str(v)    print(f"  {k:26s} {text[:70] + '...' if len(text) > 70 else text}")

---### Where to go next- Change **THE KNOBS** in section 2 and re-run it — that is the whole point of this file.- `m.replace(horizon=4)` returns a *new* measure; specs are frozen, so sweeps are safe.- Every axis is open: implement a protocol, declare your types, register a name. See  `examples/` for a POS-tag representation, a tree-edit distance, and an  entailment-clustering statistic — none of which required changing the library.